# clock/01 — controllable condition-checking dataset

Generate `check_conditions(n)` programs that check an ordered list of **independent** integer predicates and print pass/fail per condition. Eyeball LM outputs across language × modularity before wiring up labeling/attention.

Knobs (`src.goals.condition_data`): `lang` ∈ {python, cpp, java, javascript}, `modularity` ∈ {modular, monolithic}, `without_comment`, `num_conditions`. Monolithic seeds the entry signature; modular seeds empty (helpers-first, compile-safe for C++/Java).

In [1]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append("../..")

In [2]:
import os
import logging

import torch
from rich.console import Console
from rich.theme import Theme
from rich.markup import escape

from src.utils import logging_utils

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1,2,3,4,5,6,7"

console = Console(theme=Theme({"default": "white"}), highlight=False)
def rich_print(*a, **k): console.print(*a, **k)

logging_utils.configure(level=logging.INFO)

In [3]:
from src.models import ModelandTokenizer

MODEL_NAME = "Qwen/Qwen2.5-Coder-32B-Instruct"
mt = ModelandTokenizer(
    model_key=MODEL_NAME,
    dtype=torch.bfloat16,
    device_map="auto",
    attn_implementation="eager",
)
print(f"n_layer={mt.n_layer}, n_embd={mt.n_embd}, n_heads={mt.model.config.num_attention_heads}")

2026-06-30 15:14:58 src.models WARNING  Qwen/Qwen2.5-Coder-32B-Instruct not found in /disk/u/arnab/Codes/Models
If not found in cache, model will be downloaded from HuggingFace to cache directory: /disk/u/models
2026-06-30 15:14:58 httpx INFO     HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-Coder-32B-Instruct/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-06-30 15:14:58 httpx INFO     HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-Coder-32B-Instruct/381fc969f78efac66bc87ff7ddeadb7e73c218a7/config.json "HTTP/1.1 200 OK"
2026-06-30 15:14:58 httpx INFO     HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-Coder-32B-Instruct/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-06-30 15:14:58 httpx INFO     HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-Coder-32B-Instruct/381fc969f78efac66bc87ff7ddeadb7e73c218a7/tokenizer_config.json "HTTP/1.1 200 OK"
2026-06-30 15:14:58 ht

Loading weights:   0%|          | 0/771 [00:00<?, ?it/s]

2026-06-30 15:15:22 httpx INFO     HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-Coder-32B-Instruct/resolve/main/generation_config.json "HTTP/1.1 307 Temporary Redirect"
2026-06-30 15:15:22 httpx INFO     HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-Coder-32B-Instruct/381fc969f78efac66bc87ff7ddeadb7e73c218a7/generation_config.json "HTTP/1.1 200 OK"
2026-06-30 15:15:23 src.models INFO     loaded model <Qwen/Qwen2.5-Coder-32B-Instruct> | size: 62492.135 MB | dtype: torch.bfloat16 | device: cuda:0
n_layer=64, n_embd=5120, n_heads=40


## 1. Sample specs + materialize

In [4]:
from src.goals.condition_data import sample_specs, materialize, ConditionSample

### ---- config ----
LANGS         = ["python", "cpp"]      # python + one other; switch "cpp"->"java"/"javascript"
MODULARITIES  = ["modular", "monolithic"]
NUM_CONDITIONS = [3, 4, 5]
N_SPECS       = 4                       # total completions = N_SPECS * len(LANGS) * len(MODULARITIES)
SEED          = 7
MAX_NEW_TOKENS = 512
### ----------------

specs = sample_specs(n_samples=N_SPECS, num_conditions=NUM_CONDITIONS, seed=SEED)
samples = materialize(specs, langs=LANGS, modularities=MODULARITIES)
print(f"{len(specs)} specs -> {len(samples)} samples  ({LANGS} x {MODULARITIES})")
for sp in specs:
    print(f"  {sp.spec_id}  ({sp.num_conditions})  {sp.predicate_ids}")

4 specs -> 16 samples  (['python', 'cpp'] x ['modular', 'monolithic'])
  cond_0000  (4)  ['even', 'num_digits_ge', 'num_divisors_gt', 'le_bound']
  cond_0001  (3)  ['set_bits_gt', 'set_bits_gt', 'ge_bound']
  cond_0002  (3)  ['num_divisors_gt', 'num_divisors_gt', 'digit_product_gt']
  cond_0003  (4)  ['set_bits_gt', 'even', 'digit_sum_divisible', 'in_range']


## 2. Generate completions (greedy) and eyeball

In [5]:
from src.goals.functional import apply_chat_template, supports_chat_template
from src.tokens import prepare_input

def build_prompt(sample):
    instr = sample.render_instruction()
    stub  = sample.render_code_stub()
    if supports_chat_template(mt):
        return apply_chat_template(mt=mt, user_query=instr.strip(), incomplete_code=stub)
    return instr + "\nAnswer:\n" + stub

test_sample = [sample for sample in samples if sample.modularity == "monolithic"][7]
print(build_prompt(test_sample))

<|im_start|>system
You are a helpful assistant proficient in programming. When completing code, ensure proper syntax and logical correctness. Make sure to have all codes inside the <code> </code> tags.<|im_end|>
<|im_start|>user
Write a C++ program that implements a function `check_conditions` which, given an integer n, checks the following conditions:

1. n has more than 3 ones in its binary representation
2. n is even
3. the sum of n's digits is divisible by 4
4. n is between 0 and 500 (inclusive)

For each condition, in the order listed, assign the boolean result of the check to its own local variable, using single lowercase letters in order — `a` for the first condition, `b` for the second, and so on. After all conditions have been checked, use a SINGLE print statement that, using those variables, reports whether each condition passed individually and whether all of them passed collectively.
Compute each condition's boolean result inside `check_conditions` (one local variable per c

In [6]:
from src.functional import generate_with_patch

test_prompt_orig = build_prompt(test_sample)
# prefix = """    a = n % 2 == 1
#     b = n <= 50
#     sm = sum([int(dig) for dig in str(abs(n))])
# """
prefix = """    a = int(n**0.5)\n"""
test_prompt = test_prompt_orig + prefix
test_gen = generate_with_patch(
    mt=mt,
    inputs=test_prompt,
    n_gen_per_prompt=1,
    max_new_tokens=128,
    do_sample=False,
    remove_prefix=True,
)[0]

rich_print(
    f"[white]{escape(test_prompt_orig)}[/white][bright_yellow]{escape(prefix)}[/bright_yellow][bright_green]{escape(test_gen)}[/bright_green]"
)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
You have set `compile_config`, but we are unable to meet the criteria for compilation. Compilation will be skipped.


<|im_start|>system
You are a helpful assistant proficient in programming. When completing code, ensure proper syntax and logical 
correctness. Make sure to have all codes inside the <code> </code> tags.<|im_end|>
<|im_start|>user
Write a C++ program that implements a function `check_conditions` which, given an integer n, checks the following 
conditions:

1. n has more than 3 ones in its binary representation
2. n is even
3. the sum of n's digits is divisible by 4
4. n is between 0 and 500 (inclusive)

For each condition, in the order listed, assign the boolean result of the check to its own local variable, using 
single lowercase letters in order — `a` for the first condition, `b` for the second, and so on. After all 
conditions have been checked, use a SINGLE print statement that, using those variables, reports whether each 
condition passed individually and whether all of them passed collectively.
Compute each condition's boolean result inside `check_conditions` (one local variable per condition); do not define
any helper functions.
Do not include any comments in your code.<|im_end|>
<|im_start|>assistant
<code>
void check_conditions(int n) {
    a = int(n**0.5)
    bool a = __builtin_popcount(n) > 3;
    bool b = n % 2 == 0;
    bool c = (n / 100 + (n / 10) % 10 + n % 10) % 4 == 0;
    bool d = n >= 0 && n <= 500;
    printf("Condition 1: %s, Condition 2: %s, Condition 3: %s, Condition 4: %s, All Conditions: %s\n", a ? "Passed"
: "Failed", b ?

In [ ]:
print(test_prompt)

In [ ]:
from scripts.clock.score_condition_heads import load_head_scores
import numpy as np

# ── pick the scoring config to inspect ────────────────────────────────────────
Q_SPAN = 15
COMPARE_WITH = "mean"        # "mean" | "max"
TOP_K = 100
VERSION = "conditions"
# ──────────────────────────────────────────────────────────────────────────────
gaze_scores = load_head_scores(model=MODEL_NAME, version=VERSION, q_span=Q_SPAN, compare_with=COMPARE_WITH)["combined"]

# ── top-K gaze heads ──────────────────────────────────────────────────────────
flat_idx = np.argsort(gaze_scores, axis=None)[::-1][:TOP_K]
gaze_heads = [(int(l), int(h)) for l, h in (np.unravel_index(i, gaze_scores.shape) for i in flat_idx)]
print(f"top-{TOP_K} gaze heads  (compare_with='{COMPARE_WITH}', q_span={Q_SPAN})  "
      f"score range [{gaze_scores[gaze_heads[-1]]:.3f}, {gaze_scores[gaze_heads[0]]:.3f}]")
for rank, (l, h) in enumerate(gaze_heads[:15], start=1):
    print(f"  {rank:2d}. L{l:2d} H{h:2d}  score={gaze_scores[l, h]:.3f}")
if TOP_K > 15:
    print(f"  ... (+ {TOP_K - 15} more)")

In [ ]:
from src.attention import get_attention_matrices, show_interactive_attention

full_prompt = test_prompt_orig + prefix + test_gen
full_prompt = full_prompt.replace("\n", "    \n")  # more uniform tokenization on newlines
selected_heads = sorted(gaze_heads)  # or goalscope_heads
value_weighted = True

attn_info = get_attention_matrices(input = full_prompt, mt=mt, value_weighted = value_weighted)
show_interactive_attention(
    attn_info=attn_info,
    selected_heads=selected_heads,
    start_from=0,
    agg="mean",
    normalize="row_max",
    cmap="Blues",
    title=f"{len(selected_heads)} selected heads ({value_weighted=})",
    height=None,
)

In [ ]:
mt.tokenizer.bos_token_id

In [ ]:
BATCH_SIZE = 8
_pad_id = mt.tokenizer.pad_token_id if mt.tokenizer.pad_token_id is not None else mt.tokenizer.eos_token_id

@torch.no_grad()
def batched_generate(samples, max_new=MAX_NEW_TOKENS, batch_size=BATCH_SIZE):
    outs = {}
    for i in range(0, len(samples), batch_size):
        chunk = samples[i:i + batch_size]
        inputs = prepare_input(prompts=[build_prompt(s) for s in chunk], tokenizer=mt)  # left-padded
        plen = inputs["input_ids"].shape[1]
        gen = mt._model.generate(
            input_ids=inputs["input_ids"], attention_mask=inputs["attention_mask"],
            max_new_tokens=max_new, do_sample=False, pad_token_id=_pad_id,
        )
        for s, row in zip(chunk, gen[:, plen:]):  # only the new tokens; skip pad/eos on decode
            outs[(s.spec.spec_id, s.lang, s.modularity)] = mt.tokenizer.decode(row, skip_special_tokens=True)
    return outs

completions = batched_generate(samples)
print(f"generated {len(completions)} completions (batch_size={BATCH_SIZE})")

In [ ]:
# eyeball: stub + generation per (lang, modularity)
for s in samples:
    out = completions[(s.spec.spec_id, s.lang, s.modularity)]
    rich_print(f"\n[bold cyan]== {s.lang} | {s.modularity} | {s.spec.spec_id}  "
               f"preds={s.spec.predicate_ids} ==[/bold cyan]")
    rich_print(f"[grey58]{escape(' | '.join(s.spec.spec_texts))}[/grey58]")
    rich_print(f"[salmon1]{escape(s.render_code_stub() + out)}[/salmon1]")

## 3. (optional) Quick correctness peek — Python only

Run the generated Python `check_conditions` on its test inputs and compare stdout to the oracle. Eyeball only; C++/Java need compile+run and the negative-`%` oracle caveat.

In [ ]:
import io, contextlib

def peek_python(spec_id, modularity, n_show=2):
    spec = next(sp for sp in specs if sp.spec_id == spec_id)
    code = "def check_conditions(n: int):\n" if modularity == "monolithic" else ""
    code += completions[(spec_id, "python", modularity)]
    ns = {}
    try:
        exec(code, ns)
    except Exception as e:
        rich_print(f"[red]exec failed: {e}[/red]"); return
    fn = ns.get("check_conditions")
    for n in spec.test_inputs[:n_show]:
        buf = io.StringIO()
        try:
            with contextlib.redirect_stdout(buf): fn(n)
        except Exception as e:
            rich_print(f"[red]n={n}: runtime error {e}[/red]"); continue
        indiv, collective = spec.expected_results(n)
        rich_print(f"n={n}  expected individual={indiv}  all={collective}")
        rich_print(f"   stdout: {buf.getvalue().strip()!r}")

# example:
# peek_python(specs[0].spec_id, "monolithic")

## 4. Test the oracle step-labeling (Claude)

Run `label_generation` (the same oracle used by `build_multistep_clock_data`) on one generated Python sample and visualize: each condition gets a color; its **spec span in the prompt** and its **code span in the generation** share that color. Verify the oracle assigns the right code region to each condition before scaling up. (Calls the Claude API; cached.)

In [ ]:
from src.goals.multistep_label import label_generation
from rich.text import Text
from rich.table import Table

### ---- pick one generated python sample to label ----
LABEL_LANG = "cpp"
LABEL_MODULARITY = "monolithic"   # or "modular"
### ----------------------------------------------------
_demo = [s for s in samples if s.lang == LABEL_LANG and s.modularity == LABEL_MODULARITY][3]
_prompt = build_prompt(_demo)
_gen = completions[(_demo.spec.spec_id, _demo.lang, _demo.modularity)]
lab = label_generation(mt, _demo, _prompt, _gen, use_cache=True)
print(f"labeled {_demo.name}  ({len(lab.labels)} conditions)")

PALETTE = ["green", "cyan", "magenta", "yellow", "blue", "bright_red",
           "bright_cyan", "bright_magenta", "bright_green"]
def _vis(s): return s.replace("\n", "↵\n").replace("\t", "⇥")

def render_oracle_labels(lab):
    combined = lab.full_prompt + lab.generation
    enc = mt.tokenizer(combined, return_offsets_mapping=True, add_special_tokens=False)
    ids = enc["input_ids"]; offs = [tuple(p) for p in enc["offset_mapping"]]
    char_off = len(lab.full_prompt)
    gen_start = next(j for j, (a, b) in enumerate(offs) if a >= char_off and b > a)
    prompt_step, gen_step, seg_start = {}, {}, {}
    for i, sl in enumerate(lab.labels):
        if sl.prompt_token_range:
            for t in range(*sl.prompt_token_range): prompt_step[t] = i
        if sl.gen_token_range:
            for t in range(*sl.gen_token_range): gen_step[t] = i
            seg_start.setdefault(sl.gen_token_range[0], i)
    sty = lambda i: f"black on {PALETTE[i % len(PALETTE)]}"
    rich_print(f"[bold]{lab.sample_name}[/bold]   (each condition = a color; spec span + code span share it)")
    txt = Text()
    for t in range(len(ids)):
        if t == gen_start:
            txt.append("\n" + "─" * 28 + " GENERATION " + "─" * 28 + "\n", style="bold grey42")
        if t in seg_start:
            txt.append(f"\n┌── cond {seg_start[t]} ──\n", style="bold yellow")
        s = _vis(mt.tokenizer.decode([ids[t]]))
        i = prompt_step.get(t) if t < gen_start else gen_step.get(t)
        base = "grey42" if t < gen_start else "grey58"
        txt.append(s, style=(sty(i) if i is not None else base))
    console.print(txt)
    tb = Table(title="oracle step labels", show_lines=True)
    for col, w in [("cond", 4), ("condition", 34), ("prompt tok", 12),
                   ("gen lines", 10), ("gen tok", 12), ("gen_code", 40), ("reasoning", 36)]:
        tb.add_column(col, overflow="fold", max_width=w)
    for i, sl in enumerate(lab.labels):
        tb.add_row(str(i), sl.step_text, str(sl.prompt_token_range), str(sl.gen_line_range),
                   str(sl.gen_token_range), _vis(sl.gen_code), sl.reasoning)
    console.print(tb)

render_oracle_labels(lab)

## 5. Build + save the full dataset (1000 samples)

500 specs (k ∈ {3,4,5}) × {modular, monolithic} = **1000 matched samples**, Python. Matched pairs (same spec across modularities) give the clean implementation-invariance contrast. Saved to `data/multistep/` for `scripts/clock/build_condition_data.py`.

In [ ]:
# from src.goals.condition_data import sample_specs, materialize, ConditionDataset
# from src.utils import env_utils
# from pathlib import Path
# from collections import Counter

# ### ---- full dataset config ----
# DS_N_SPECS        = 500                       # x {modular, monolithic} = 1000 samples
# DS_NUM_CONDITIONS = [3, 4, 5]
# DS_LANGS          = ["python"]
# DS_MODULARITIES   = ["modular", "monolithic"]
# DS_SEED           = 0
# DS_OUT = Path(env_utils.DEFAULT_DATA_DIR) / "multistep" / "condition_check_python.json"
# ### ------------------------------

# ds_specs = sample_specs(DS_N_SPECS, DS_NUM_CONDITIONS, seed=DS_SEED)
# ds_samples = materialize(ds_specs, langs=DS_LANGS, modularities=DS_MODULARITIES)
# dataset = ConditionDataset(samples=ds_samples, meta={
#     "n_specs": DS_N_SPECS, "num_conditions": DS_NUM_CONDITIONS,
#     "langs": DS_LANGS, "modularities": DS_MODULARITIES, "seed": DS_SEED,
# })
# saved = dataset.save(DS_OUT)
# print(f"saved {len(ds_samples)} samples ({DS_N_SPECS} specs x {DS_MODULARITIES}) -> {saved}")
# print("modularity:", Counter(s.modularity for s in ds_samples))
# print("num_conditions:", Counter(s.spec.num_conditions for s in ds_samples))

# # build attention + labels with:
# #   python scripts/clock/build_condition_data.py --dataset {DS_OUT} --version conditions
# # then score heads with:
# #   python scripts/clock/score_condition_heads.py --version conditions --q_span 20 --compare_with max